In [1]:
import os
os.chdir('D:\\ai-engineering-buildcamp-codespace\\03-agents\\02-agent-basics\\')
print(os.getcwd())

D:\ai-engineering-buildcamp-codespace\03-agents\02-agent-basics


In [2]:
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import urlparse
from markitdown import MarkItDown
from typing import Any, Dict, Iterable, List
import json
from pydantic import BaseModel, Field
from typing import Literal
from minsearch import AppendableIndex
from gitsource import GithubRepositoryDataReader, chunk_documents

In [3]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

In [4]:
from openai import OpenAI
openai_client  = OpenAI()

In [5]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [6]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [7]:
# tool_call Structure:

# tool_call.name → "search" (the function name)
# tool_call.arguments → '{"query": "your question"}' (JSON string of arguments)
# tool_call.call_id → unique identifier for the call

def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name  ## the name of the tool to call, e.g., "search".

    if name == 'search':
        result = search(**arguments)
    # elif name == 'add_entry':
    #     result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [8]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [9]:
question = "How do I create a dahsbord in Evidently?"

In [10]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [11]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

In [12]:
#For agentic workflows where responses contain multiple messages (assistant + tool results), 
# you almost always want extend() to keep the conversation history flat!
## if you use append(), you'll end up with nested lists which will break the format expected by the model in subsequent calls.
## openai client calls the tool using the make_call function, which executes the search and returns the results in the expected format.
message_history.extend(response.output)

In [13]:
# as mentioned in the instructions, the response will contain multiple search calls,
# openai_client.chat.completions.create() only suggests which tools to call
# It returns ResponseFunctionToolCall objects that describe what to call and with what arguments, but it doesn't execute the calls itself. 
# this is called parallel tool calling, where the client suggests multiple calls and it's up to you to execute them and return the results in the expected format.     
response.output

[ResponseFunctionToolCall(arguments='{"query":"create dashboard"}', call_id='call_mTUsYsKoc8Js30LUhuklNXA7', name='search', type='function_call', id='fc_08b5edece52300c70069f161d238cc8197b4f20399eefb54ed', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"dashboard examples"}', call_id='call_4fM85JHViD28fkkv0AK9OJET', name='search', type='function_call', id='fc_08b5edece52300c70069f161d238e48197a2cc763652ccaee1', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"dashboard features"}', call_id='call_IMeNRMu9KbKfg0lja5LLZiZ1', name='search', type='function_call', id='fc_08b5edece52300c70069f161d238f08197a16a0cb175b038e3', namespace=None, status='completed')]

In [14]:
## from the ealier suggessted calls we check the type of each message in the response output, and if it's a 'function_call',
#  we execute it using the make_call function.
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)  # Returns a SINGLE message object
        message_history.append(tool_call_output)  # Add that ONE message

executing search({"query":"create dashboard"})...
executing search({"query":"dashboard examples"})...
executing search({"query":"dashboard features"})...


In [15]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

response.usage.input_tokens

12027

In [16]:
print(response.output_text)

To create a dashboard in Evidently, you can follow these steps. The process can be done either through the user interface (UI) or programmatically using the Python API.

### Creating a Dashboard Using the User Interface

1. **Access Edit Mode**:
   - Go to your dashboard and enter "Edit" mode (click the top right corner).

2. **Add Tabs**:
   - You can add multiple tabs to organize your panels.
   - Click the plus sign next to "Add Tab" and select "empty" to create a custom tab or use pre-built templates.

3. **Add Panels**:
   - Click on the "Add Panel" button.
   - Follow the prompts to configure the panel.
   - You'll need to select the appropriate metrics that match those in your logged reports.
   - Use filters (like tags) to refine the data shown in your panels.
   - After setting it up, click "Save" and select the desired tab for the new panel.

4. **Delete Tabs/Panels**:
   - Enter "Edit" mode again to manage tabs or panels. Use the edit option to delete as needed.

### Creatin

Lets try to make Agentic Rag iterative, where the model can call tools multiple times in a loop until it has enough information to answer the question.

In [17]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [18]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break


# iteration number 1...
# ASSISTANT: I'll search for dashboard creation in the documentation.
# executing search({"query": "create dashboard"})...

# iteration number 2...
# ASSISTANT: Based on the initial search, I found info about dashboards. Now I'll search for: 1) dashboard configuration details, 2) dashboard examples
# executing search({"query": "dashboard configuration"})...
# executing search({"query": "dashboard examples"})...

# iteration number 3...
# ASSISTANT: Based on the searches, here's how to create a dashboard in Evidently:

# 1. Import the Dashboard class from evidently.ui.dashboards
# 2. Configure your metrics using MetricPreset or custom metrics
# 3. Generate the dashboard using dashboard.calculate(reference_data, current_data)

# Here's a complete example:
# [Full code example]

# This approach allows you to create interactive dashboards for monitoring your ML models.

iteraration number 1...
executing search({"query":"create dashboard"})...

iteraration number 2...
executing search({"query":"Dashboard management"})...
executing search({"query":"dashboard examples"})...

iteraration number 3...
ASSISTANT: To create a dashboard in Evidently, you can follow these general steps:

1. **Create a Project**: Before creating a dashboard, ensure that you have set up a project and added relevant reports that contain evaluation results.

2. **Dashboard Creation**:
   - **In the User Interface**:
     - Navigate to the dashboard section and enter "Edit" mode.
     - Use the "Add Tab" function to organize your dashboard. You can create multiple tabs as needed.
     - Click "Add Panel" to insert various visualizations such as text panels, counters, pie charts, line plots, and bar plots. Configure each panel by selecting metrics, which correspond to the reports you have logged.
     - Save your changes after arranging your panels.

   - **Using Python API**:
     -

 Message History Evolution
Iteration 1:
 [system, user, assistant_msg, function_call, tool_result]

Iteration 2:
[system, user, assistant_msg_1, function_call_1, tool_result_1,
 assistant_msg_2, function_call_2, function_call_3, tool_result_2, tool_result_3]

Iteration 3:

[system, user, ...(all previous)..., final_answer]

Adding an new tool called add_entry
Purpose of add_entry Tool
The add_entry tool allows the agent to ADD NEW DOCUMENTATION to the knowledge base, not just search it.

Before (Search-only agent):
Agent can only READ from the knowledge base
↓
search() → Find existing docs

After (Search + Add agent):

Agent can READ and WRITE to the knowledge base
↓
search() → Find existing docs
add_entry() → Create new docs

In [19]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [ ]:
# The make_call function is the central point where the agent's suggested tool calls are executed.
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [20]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To assist you with creating a dashboard, I'll first search for information on dashboard creation processes. This will help me understand the necessary steps and components involved in creating a dashboard within the tool. 

### First Search
I'll look up "creating a dashboard" to find relevant guidelines and instructions.
executing search({"query":"creating a dashboard"})...

iteraration number 2...
ASSISTANT: The first search provided information about how to create dashboards, including the process of adding tabs and panels. To create a dashboard, you need to enter "Edit" mode, then you can add tabs and various types of panels for visualizing evaluation results.

### Summary from First Search:
1. **Adding Tabs**: 
   - Enter "Edit" mode.
   - Click the plus sign to add a tab.
   - Provide a name for the tab or choose a pre-built template.
   
2. **Adding Panels**: 
   - Again, enter "Edit" mode, and click "Add Panel".
   - Configure the panel setting

In [21]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)


In [22]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break

iteraration number 5...
executing add_entry({"filename":"docs/platform/dashboard_creation.mdx","title":"Creating a Dashboard","description":"Step-by-step guide on creating and managing dashboards.","content":"### Creating a Dashboard\nTo create a dashboard, you typically follow these steps:\n\n1. **Add Tabs**: Organize your dashboard with multiple tabs for better layout.\n   - Start by entering \"Edit\" mode and click to add a new tab.\n\n2. **Add Panels**: \n   - You can add various panels (text, counters, pie charts, etc.) either via the UI or the Python API.\n   - Configure each panel by selecting the appropriate metrics, visualization types, and settings for how you want the data represented.\n\n3. **Customization**: \n   - Use filtering and aggregation options to tailor the dashboard to your specific needs.\n\n### Using the Python API\n- To manage dashboards, you need to create a project and connect to the relevant cloud instance.\n- **Adding Tabs**: You can create a new tab with 

In [23]:
index.docs[-1]

{'start': 0,
 'content': "When working on an AI system, you need test data to run automated evaluations for quality and safety. A test dataset is a structured set of test cases. It can contain:\n\n* Just the inputs, or\n* Both inputs and expected outputs (ground truth).\n\nYou can use this test dataset to:\n\n* Run **experiments** and track if changes improve or degrade system performance.\n* Run **regression testing** to ensure updates don’t break what was already working.\n* **Stress-test** your system with complex or adversarial inputs to check its resilience.\n\n![](/images/synthetic/synthetic_experiments_img.png)\n\nYou can create test datasets manually, collect them from real or historical data, or generate them synthetically. While real data is best, it is not always available or sufficient to cover all cases. Public LLM benchmarks help with general model comparisons but don’t reflect your specific use case. Manually writing test cases takes time and effort.\n\n**Synthetic data 